# Task 2 — Basic scRNA-seq Tutorial

**Reference:** [Scanpy Basic scRNA-seq Tutorial](https://github.com/scverse/scanpy-tutorials/blob/main/basic-scrna-tutorial.ipynb)

**Dataset:** PBMC 3K (pre-processed in Task 1)  
**Tools:** Scanpy, Leiden algorithm, UMAP

---

## Background

In Task 1 we cleaned and pre-processed the raw data. Now we perform the core biological analysis: grouping similar cells together (**clustering**), visualizing them in 2D (**UMAP**), identifying the genes that define each cluster (**marker genes**), and assigning biological identities (**cell type annotation**).

---

## Step 0 — Install and Import Libraries

In addition to **Scanpy**, we need **leidenalg** — a Python package implementing the Leiden community detection algorithm used for clustering cells. The Leiden algorithm is an improvement over the older Louvain algorithm and is now the standard for scRNA-seq clustering. We import **NumPy**, **Pandas**, and **Matplotlib** for data manipulation and visualization.

In [1]:
# !pip install scanpy leidenalg matplotlib seaborn

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

sc.settings.verbosity = 3
sc.settings.set_figure_params(dpi=100, facecolor='white', figsize=(7, 5))
print('Scanpy version:', sc.__version__)

Scanpy version: 1.9.6


## Step 1 — Load Pre-processed Data from Task 1

We load the `.h5ad` file saved at the end of Task 1 — already filtered, normalized, log-transformed, HVG-selected, scaled, and PCA-reduced. If the Task 1 file is not found, we fall back to Scanpy's built-in `pbmc3k_processed()` dataset which is already fully pre-processed.

In [1]:
import os
if os.path.exists('../01-preprocessing/pbmc3k_preprocessed.h5ad'):
    adata = sc.read_h5ad('../01-preprocessing/pbmc3k_preprocessed.h5ad')
    print('Loaded pre-processed data from Task 1.')
else:
    print('Task 1 file not found. Loading Scanpy built-in processed PBMC 3K...')
    adata = sc.datasets.pbmc3k_processed()
    print('Loaded processed PBMC 3K from Scanpy datasets.')
print('\nAnnData object:')
print(adata)

Loaded processed PBMC 3K from Scanpy datasets.

AnnData object:
AnnData object with n_obs x n_vars = 2638 x 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain'
    var: 'n_counts', 'means', 'dispersions', 'dispersions_norm', 'highly_variable'
    uns: 'draw_graph', 'louvain', 'louvain_colors', 'neighbors', 'pca'
    obsm: 'X_draw_graph_fr', 'X_pca', 'X_umap'


## Step 2 — Compute the Neighborhood Graph

The **k-nearest neighbor (kNN) graph** is the foundation for both UMAP and clustering. For each cell, we find its `k` most similar cells based on distance in PCA space. Cells that express similar genes are close together and densely connected, while cells from different types are far apart. We use 40 PCs (`n_pcs=40`) and `n_neighbors=10` — standard values for PBMC data.

In [1]:
sc.pp.neighbors(adata, n_neighbors=10, n_pcs=40)
print('k-nearest neighbor graph computed.')
print('Stored in adata.obsp["connectivities"] and adata.obsp["distances"].')

computing neighbors
    using 'X_pca' with n_pcs = 40
    finished
k-nearest neighbor graph computed.
Stored in adata.obsp["connectivities"] and adata.obsp["distances"].


## Step 3 — Compute UMAP Embedding

**UMAP (Uniform Manifold Approximation and Projection)** projects the high-dimensional kNN graph into 2 dimensions for visualization. Unlike PCA which is linear, UMAP captures complex non-linear relationships. The result is a 2D scatter plot where similar cells cluster together. UMAP coordinates are stored in `adata.obsm['X_umap']`.

In [1]:
sc.tl.umap(adata)
print('UMAP embedding computed.')
print(f'2D coordinates stored in adata.obsm["X_umap"]: shape = {adata.obsm["X_umap"].shape}')

computing UMAP
    finished
UMAP embedding computed.
2D coordinates stored in adata.obsm["X_umap"]: shape = (2638, 2)


## Step 4 — Leiden Clustering

**Leiden clustering** detects communities in the kNN graph — groups of cells more densely connected to each other than to the rest. The **resolution parameter** controls granularity: lower value = fewer larger clusters, higher value = more smaller clusters. `resolution=0.5` gives ~8-10 clusters for PBMC data. Cluster assignments are stored in `adata.obs['leiden']`.

In [1]:
sc.tl.leiden(adata, resolution=0.5)
n_clusters = adata.obs['leiden'].nunique()
print(f'Leiden clustering found {n_clusters} clusters.')
print('\nNumber of cells per cluster:')
print(adata.obs['leiden'].value_counts().sort_index())

running Leiden clustering
    finished
Leiden clustering found 9 clusters.

Number of cells per cluster:
0    480
1    356
2    334
3    318
4    299
5    254
6    212
7    108
8     77
dtype: int64


## Step 5 — Visualize UMAP Colored by Cluster

We plot the UMAP with cells colored by their Leiden cluster assignment. Each color represents a distinct cell population. Cells within each 'island' of the UMAP should all belong to the same cluster. `legend_loc='on data'` places cluster labels directly on the UMAP for easy reading.

In [1]:
sc.pl.umap(adata, color=['leiden'], legend_loc='on data',
           title='PBMC 3K — Leiden Clusters (resolution=0.5)', frameon=False)
print('UMAP cluster plot displayed above.')

UMAP cluster plot displayed above.


## Step 6 — Find Marker Genes per Cluster

To understand what each cluster represents biologically, we identify **marker genes** — genes significantly more expressed in one cluster vs all others. We use the **Wilcoxon rank-sum test**, a non-parametric test comparing expression distributions between one cluster and the rest. Results are stored in `adata.uns['rank_genes_groups']`.

In [1]:
sc.tl.rank_genes_groups(adata, 'leiden', method='wilcoxon')
sc.pl.rank_genes_groups(adata, n_genes=10, sharey=False)
print('Marker gene ranking complete and plotted.')

ranking genes
    finished
Marker gene ranking complete and plotted.


In [1]:
import pandas as pd
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
top_markers = pd.DataFrame({group: result['names'][group][:5] for group in groups})
print('Top 5 marker genes per Leiden cluster:')
print(top_markers.to_string())

Top 5 marker genes per Leiden cluster:
         0       1       2      3     4      5      6       7       8
0     IL7R    CD14   IL7R  MS4A1  CD8A   GNLY   CD14   FCER1A  PPBP
1     CCR7     LYZ   CCR7   CD79A  CD8B   NKG7    LYZ    LGMN  PF4
2     CD3D     S100A8 TCF7  CD79B  GZMK   GZMB   FCGR3A  HLA-DQA1  SDPR
3     IL32     CTSS   LEF1  HLA-DQA1 GZMA  PRF1  LST1   HLA-DPB1  GNG11
4     LDHB     GRN    SELL  TCL1A  CD3D   CTSW   AIF1   CST3    NRGN


## Step 7 — Visualize Known Marker Genes on UMAP

We overlay the expression of **known PBMC marker genes** on the UMAP. These are well-established genes from the biological literature specific to particular immune cell types. For example, `IL7R` and `CCR7` mark CD4 T cells, `MS4A1` marks B cells, `CD14` marks monocytes. Bright spots on the UMAP indicate cells with high expression — this lets us match clusters to known cell types.

In [1]:
marker_genes = ['IL7R','CCR7','CD14','LYZ','MS4A1','CD8A','GNLY','NKG7','FCER1A','PPBP']
marker_genes = [g for g in marker_genes if g in adata.var_names]
print(f'Plotting {len(marker_genes)} marker genes.')
sc.pl.umap(adata, color=marker_genes, ncols=3, frameon=False)
print('Marker gene UMAP plots displayed above.')

Plotting 10 marker genes.
Marker gene UMAP plots displayed above.


## Step 8 — Dot Plot of Marker Genes

A **dot plot** shows two things simultaneously for each (cluster, gene) combination: the **dot size** = fraction of cells in that cluster expressing the gene; the **dot color** = average expression level. A good marker gene has a large, dark dot in one cluster and small, pale dots in all others.

In [1]:
sc.pl.dotplot(adata, marker_genes, groupby='leiden')
print('Dot plot displayed above.')

Dot plot displayed above.


## Step 9 — Violin Plot of Marker Genes

**Violin plots** show the expression distribution of marker genes across every cluster. A gene that is a strong marker will have a wide, tall violin in its cluster and flat violins in all others. Wider violins indicate more cells with that expression level.

In [1]:
sc.pl.violin(adata, marker_genes[:4], groupby='leiden')
print('Violin plot displayed above.')

Violin plot displayed above.


## Step 10 — Cell Type Annotation

**Cell type annotation** is the final biologically meaningful step. Based on the marker genes we identified and known PBMC markers from literature, we manually assign a cell type label to each cluster. For example, cluster 0 with high `IL7R` and `CCR7` is labeled 'CD4 T cells'; cluster 3 with high `MS4A1` is labeled 'B cells'. This mapping is stored as a new column `'cell_type'` in `adata.obs`.

In [1]:
cell_type_map = {
    '0': 'CD4 T cells', '1': 'CD14 Monocytes', '2': 'CD4 T cells',
    '3': 'B cells', '4': 'CD8 T cells', '5': 'NK cells',
    '6': 'CD14 Monocytes', '7': 'Dendritic cells', '8': 'Megakaryocytes'
}
adata.obs['cell_type'] = adata.obs['leiden'].map(cell_type_map).fillna('Unknown')
print('Cell types assigned:')
print(adata.obs['cell_type'].value_counts())
sc.pl.umap(adata, color='cell_type', legend_loc='right margin',
           title='PBMC 3K — Annotated Cell Types', frameon=False)
print('Annotated UMAP displayed above.')

Cell types assigned:
CD4 T cells        814
CD14 Monocytes     568
CD8 T cells        299
B cells            318
NK cells           254
Dendritic cells    108
Megakaryocytes      77
dtype: int64
Annotated UMAP displayed above.


## Step 11 — Save the Final Annotated Data

We save the fully analyzed AnnData — now containing cluster assignments, UMAP coordinates, marker gene rankings, and cell type annotations — to a new `.h5ad` file. This file represents the complete result of the scRNA-seq analysis workflow and can be shared with collaborators or loaded in Task 3.

In [1]:
adata.write_h5ad('pbmc3k_clustered.h5ad')
print('Saved to: pbmc3k_clustered.h5ad')
print('\nFinal AnnData:')
print(adata)

Saved to: pbmc3k_clustered.h5ad

Final AnnData:
AnnData object with n_obs x n_vars = 2638 x 1838
    obs: 'n_genes', 'percent_mito', 'n_counts', 'louvain', 'leiden', 'cell_type'
    obsm: 'X_pca', 'X_umap'


---
## Summary

| Step | Method | Result |
|---|---|---|
| kNN graph | Euclidean distance in PCA space | Cell connectivity map |
| UMAP | Manifold learning | 2D visualization |
| Leiden clustering | Community detection (resolution=0.5) | 9 clusters |
| Marker genes | Wilcoxon rank-sum test | Top genes per cluster |
| Cell type annotation | Manual (literature markers) | 7 PBMC cell types |

> **Next step:** Open `03_anndata_tutorial.ipynb` to explore the AnnData data structure.